In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1785250991557_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
spark = SparkSession.builder \
    .appName("Yelp RAG Pipeline") \
    .getOrCreate()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
BUSINESS_PATH = "s3://yelpdatasetsmvita/gold_layer/rag_new/business_documents/"
REVIEW_PATH = "s3://yelpdatasetsmvita/gold_layer/rag_new/review_documents/"

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
business_df = spark.read.parquet(BUSINESS_PATH)
review_df = spark.read.parquet(REVIEW_PATH)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
#Define Chunk Parameters

In [6]:
CHUNK_SIZE = 1000
OVERLAP = 200
STEP = CHUNK_SIZE - OVERLAP

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# Chunk Function

# We'll use a Spark UDF that returns an array of text chunks.

In [8]:
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType

def chunk_text(text):
    if text is None:
        return []

    chunks = []

    for i in range(0, len(text), STEP):
        chunks.append(text[i:i + CHUNK_SIZE])

        if i + CHUNK_SIZE >= len(text):
            break

    return chunks

chunk_udf = udf(chunk_text, ArrayType(StringType()))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [9]:
#Create Chunks: For the business dataset

business_chunks = business_df.withColumn(
    "chunks",
    chunk_udf("document_text")
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
#For the review dataset:

review_chunks = review_df.withColumn(
    "chunks",
    chunk_udf("document_text")
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
#Explode Chunks: Each element of the chunk array becomes a separate row.
#Business

business_chunks = business_chunks.select(
    "*",
    F.posexplode("chunks").alias("chunk_number", "chunk_text")
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [14]:
#Review:

review_chunks = review_chunks.select(
    "*",
    F.posexplode("chunks").alias("chunk_number", "chunk_text")
)

#Now every chunk is an individual row.

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [15]:
#review
business_chunks = business_chunks.withColumn(
    "chunk_id",
    F.concat_ws(
        "_",
        F.col("document_id"),
        F.col("chunk_number")
    )
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [16]:
#review
review_chunks = review_chunks.withColumn(
    "chunk_id",
    F.concat_ws(
        "_",
        F.col("document_id"),
        F.col("chunk_number")
    )
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [17]:
#Create Chunk ID, Business:

business_chunks = business_chunks.select(
    "chunk_id",
    "document_id",
    "business_id",
    "business_name",
    "city",
    "state",
    "primary_category",
    "business_rating",
    "review_count",
    "document_type",
    "chunk_number",
    "chunk_text"
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [18]:
#Create a chunk ID, Review:

review_chunks = review_chunks.select(
    "chunk_id",
    "document_id",
    "review_id",
    "business_id",
    "business_name",
    "city",
    "state",
    "primary_category",
    "stars",
    "sentiment",
    "document_type",
    "chunk_number",
    "chunk_text"
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [19]:
business_chunks.show(5, truncate=False)

review_chunks.show(5, truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------------------+------------------------------+----------------------+----------------------+------------+-----+----------------------+---------------+------------+-------------+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|chunk_id                        |document_id                  

In [20]:
print("Business Chunks:", business_chunks.count())
print("Review Chunks:", review_chunks.count())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Business Chunks: 150347
Review Chunks: 8729681

Step 3 — Write Chunked Data to S3
Objective

Persist the chunked datasets as Parquet files in S3 so they become the source for embedding generation.
At this point, the EMR job's responsibility is almost complete.

Recommended Folder Structure
gold_layer/
└── rag_new/
    ├── business_documents/
    ├── review_documents/
    │
    ├── business_chunks/
    │      part-xxxxx.parquet
    │
    └── review_chunks/
           part-xxxxx.parquet

I recommend separate folders instead of a single chunked_documents folder because:

easier maintenance
independent processing
easier debugging
easier incremental updates
cleaner retrieval pipeline

In [21]:
#Step 3.1 Define Output Paths

BUSINESS_CHUNK_PATH = "s3://yelpdatasetsmvita/gold_layer/rag_new/business_chunks/"
REVIEW_CHUNK_PATH = "s3://yelpdatasetsmvita/gold_layer/rag_new/review_chunks/"

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [22]:
# Step 3.2 Optimize Partitions

# Before writing, repartition the data.
# Business
# Around 150k records.

business_chunks = business_chunks.repartition(8)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [23]:
#Review, Around 7 million records.
review_chunks = review_chunks.repartition(64)

#The exact number depends on your EMR cluster size, 
#but 64 is a reasonable starting point.

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [24]:
#Step 3.3 Write Business Chunks

business_chunks.write \
    .mode("overwrite") \
    .parquet(BUSINESS_CHUNK_PATH)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [25]:
#Step 3.4 Write Review Chunks

review_chunks.write \
    .mode("overwrite") \
    .parquet(REVIEW_CHUNK_PATH)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [26]:
#Step 3.5 Validate
business_chunk_df = spark.read.parquet(BUSINESS_CHUNK_PATH)

review_chunk_df = spark.read.parquet(REVIEW_CHUNK_PATH)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [27]:
#Check counts

print(business_chunk_df.count())
print(review_chunk_df.count())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

150347
8729681

In [28]:
#Check schema
business_chunk_df.printSchema()
review_chunk_df.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- chunk_id: string (nullable = true)
 |-- document_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- business_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- primary_category: string (nullable = true)
 |-- business_rating: double (nullable = true)
 |-- review_count: long (nullable = true)
 |-- document_type: string (nullable = true)
 |-- chunk_number: integer (nullable = true)
 |-- chunk_text: string (nullable = true)

root
 |-- chunk_id: string (nullable = true)
 |-- document_id: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- business_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- primary_category: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- sentiment: string (nullable = true)
 |-- document_type: string (nullable = true)
 |-- chunk_number: i

In [29]:
#View samples

business_chunk_df.show(5, truncate=False)
review_chunk_df.show(5, truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------------------+------------------------------+----------------------+--------------------------------+--------------+-----+----------------------+---------------+------------+-------------+------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|chunk_id                        |document_id                   |business_id           |business_name     

Why save chunked data?

Because chunking is an expensive preprocessing step.

If you later:

change embedding models,
compare embedding models,
rebuild the FAISS index,
migrate to another vector database,

you do not need to chunk the documents again. You simply reuse the stored chunked Parquet files.

Recommendation

Since you're already using EMR, I recommend keeping the embedding generation on EMR. This gives you an end-to-end AWS data pipeline:

In [ ]:
#Step 4 — Embedding Generation
#Goal: Convert every chunk_text into a dense numerical vector

# Example: 
#Chunk
# "The pizza was amazing..."
# ↓
# Embedding Model
# ↓
# [-0.182,
#  0.337,
#  ...
#  0.921]

#This vector captures the semantic meaning of the text.

Choosing the Embedding Model

I recommend:

BAAI/bge-small-en-v1.5

Why?

Excellent retrieval quality.
Lightweight (~130 MB).
384-dimensional embeddings.
Fast on CPU.
Widely used in production RAG systems.

In [ ]:
#---------------------------------

Install Required Libraries
On the EMR master node:

In [30]:
print(spark.sparkContext.defaultParallelism)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2